# Synapse submission — batch-size experiment (BN@8 and GN@8, LeakyReLU)

Adapted from `Synapse_Submission_IN_8Activations_v4.ipynb`. Same pipeline, same predictor
settings, same uint8/zip packaging — but for the **two batch-8 cells** trained on RunPod for
the batch-size question, read from **`temp_nnunet_results`**.

| label | trainer | role |
|---|---|---|
| `BN_LeakyReLU_bs8` | `nnUNetTrainer_500ep_BN_LeakyReLU_bs8` | condition under test |
| `GN_LeakyReLU_bs8` | `nnUNetTrainer_500ep_GN_LeakyReLU_bs8` | batch-independent control |

**Output:** one Synapse zip per cell → 2 zips, scored on the same 219-case blinded validation
set as every other cell, so the paired Wilcoxon against the batch-2 counterparts is valid.

### What changed from the 8-activation notebook

- **Trainer file is `nnUNetTrainer_batchsize.py`** — one self-contained file (activations,
  norm swap, batch override all inside). It defines `build_network_architecture` as a
  `@classmethod`, so nnU-Net's class-level call at inference works **without the runtime patch**
  the earlier notebook needed. Cell 2 verifies this by building each network and counting modules.
- **Model folders are resolved by exact trainer name**, not by activation token — both cells use
  LeakyReLU, so token matching would be ambiguous.
- **Norm census, not just activation census.** For each cell the verification counts
  `BatchNorm3d` / `GroupNorm` / `InstanceNorm3d` modules: the BN cell must show BatchNorm3d only,
  the GN cell GroupNorm only, and both must show 44 LeakyReLU. A wrong norm here means the
  checkpoint would load into the wrong architecture and fail — or worse, load silently.
- **Batch size does not matter at inference.** The `_bs8` trainers override the training batch
  in `initialize()`, which the predictor never calls. Inference is sliding-window over one case at
  a time, identical to the batch-2 cells.
- Everything else (staging, predictor, zip, manifest) is unchanged so results stay comparable.


## 0 — Install, then RESTART the runtime  ⚠️

**Run this cell first, on its own. It will restart the session. That is expected.**

`pip install nnunetv2==2.2.1` pulls in NumPy 2.5.x, which breaks two ways:

- The already-imported NumPy in the live kernel is stale relative to the newly installed
  Python-level code, giving
  `AttributeError: module 'numpy._core._multiarray_umath' has no attribute '_blas_supports_fpe'`
  on `import nnunetv2...`. Only a kernel restart clears that.
- `numba` (a transitive nnU-Net dependency) requires `numpy<2.1`, so 2.5.x is genuinely
  incompatible, not merely stale.

So NumPy is pinned to `<2.1` and the runtime is restarted. **After the restart, continue from
cell 1 — do not re-run this cell** (it is idempotent and will simply report "already satisfied",
but re-running wastes a minute).

In [ ]:
import importlib.util, sys

def _numpy_ok():
    try:
        import numpy as _np
    except Exception:
        return False
    major, minor = (int(x) for x in _np.__version__.split(".")[:2])
    return (major, minor) < (2, 1)

already = importlib.util.find_spec("nnunetv2") is not None and _numpy_ok()

if already:
    import numpy as _np
    print(f"Already set up — nnunetv2 present, numpy {_np.__version__} (<2.1). No restart needed.")
else:
    print("Installing nnU-Net with numpy pinned <2.1 (numba requires it)...")
    !pip install -q "numpy<2.1" nnunetv2==2.2.1 nibabel
    print("\nInstall done. Restarting the runtime so the pinned NumPy is loaded cleanly.")
    print("After it restarts, run cell 1 next — skip this cell.")
    import IPython
    IPython.Application.instance().kernel.do_shutdown(True)

## 1 — Mount & configure

`RESULTS_ROOT` is where the two bs8 models live (copied from the pod with rclone). The cell tries
the likely locations for `temp_nnunet_results`; override by hand if none match.


In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive', force_remount=True)

WORKSPACE = "/content/drive/MyDrive/nnU-Net Project/workspace"
DATASET   = "Dataset100_BraTS2023"

# --- the two batch-8 cells: label -> trainer class name (== results folder prefix) ---
CELLS = {
    "BN_LeakyReLU_bs8": "nnUNetTrainer_500ep_BN_LeakyReLU_bs8",
    "GN_LeakyReLU_bs8": "nnUNetTrainer_500ep_GN_LeakyReLU_bs8",
}
NORM_OF = {"BN_LeakyReLU_bs8": "BN", "GN_LeakyReLU_bs8": "GN"}
LABELS  = list(CELLS)

# --- where the trained models are: temp_nnunet_results ---
_candidates = [
    "/content/drive/MyDrive/temp_nnunet_results",
    os.path.join(WORKSPACE, "temp_nnunet_results"),
    "/content/drive/MyDrive/nnU-Net Project/temp_nnunet_results",
]
RESULTS_ROOT = next((p for p in _candidates if os.path.isdir(p)), None)

print("Drive mounted.")
for p in _candidates:
    print(f"   {'[FOUND]' if os.path.isdir(p) else '[     ]'} {p}")
if RESULTS_ROOT is None:
    print("\n   None of the candidates exist. Listing MyDrive top level:")
    for d in sorted(os.listdir("/content/drive/MyDrive"))[:60]:
        print("     -", d)
    raise FileNotFoundError("Set RESULTS_ROOT manually to the temp_nnunet_results folder.")
print(f"\n   RESULTS_ROOT = {RESULTS_ROOT}")

# --- nnU-Net environment ---
os.environ["nnUNet_raw"]          = os.path.join(WORKSPACE, "nnUNet_raw")
os.environ["nnUNet_preprocessed"] = os.path.join(WORKSPACE, "nnUNet_preprocessed")
os.environ["nnUNet_results"]      = RESULTS_ROOT

RAW_VAL_DIR      = os.path.join(WORKSPACE, "ASNR-MICCAI-BraTS2023-GLI-Challenge-ValidationData")
TARGET_IMAGES_TS = os.path.join(os.environ["nnUNet_raw"], DATASET, "imagesTs")
PRED_ROOT        = os.path.join(WORKSPACE, "Ensemble_Predictions")
ZIP_ROOT         = os.path.join(WORKSPACE, "Synapse_Submissions_bs8")
os.makedirs(TARGET_IMAGES_TS, exist_ok=True)
os.makedirs(PRED_ROOT, exist_ok=True)
os.makedirs(ZIP_ROOT, exist_ok=True)

print(f"   raw val:  {RAW_VAL_DIR}")
print(f"   preds ->  {PRED_ROOT}")
print(f"   zips  ->  {ZIP_ROOT}")
print(f"   plan:     {len(LABELS)} cells {LABELS}")


## 2 — Setup, trainer injection, model discovery, and architecture verification

Applies the PyTorch `weights_only` patch, copies **only** `nnUNetTrainer_batchsize.py` from
`workspace/trainer files/` into the nnU-Net package, resolves each cell's results folder by exact
trainer name, then **builds each network the way the predictor will** and counts normalisation and
activation modules.

Only one trainer file is injected. A single unimportable module in nnU-Net's trainer directory
makes its class resolver fail for *every* class with a misleading error — copying the older
`custom_brats_activations_remaining.py` / `nnUNetTrainer_IN_remaining.py` alongside would
reintroduce that risk for no benefit; the bs8 trainers do not import from them.


In [ ]:
import os, shutil, glob, sys, re
import numpy as np

_maj, _min = (int(x) for x in np.__version__.split(".")[:2])
if (_maj, _min) >= (2, 1):
    raise RuntimeError(f"numpy {np.__version__} is too new (numba needs <2.1). Run cell 0 and let it restart.")
print(f"numpy {np.__version__} OK")

from tqdm import tqdm
import torch
import nnunetv2
from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor
from nnunetv2.utilities.find_class_by_name import recursive_find_python_class
from nnunetv2.utilities.plans_handling.plans_handler import PlansManager
from batchgenerators.utilities.file_and_folder_operations import load_json, join
print(f"nnunetv2 {getattr(nnunetv2, '__version__', '2.2.1')} imported OK")

# --- PyTorch weights_only patch (idempotent) ---
if not getattr(torch.load, "_weights_only_patch", False):
    _orig_load = torch.load
    def _patched_load(*args, **kwargs):
        kwargs["weights_only"] = False
        return _orig_load(*args, **kwargs)
    _patched_load._weights_only_patch = True
    torch.load = _patched_load
    print("torch.load patched (weights_only=False).")

TRAINER_DIR = os.path.join(os.path.dirname(nnunetv2.__file__), "training", "nnUNetTrainer")
TRAINER_PKG = "nnunetv2.training.nnUNetTrainer"

# --- inject the ONE trainer file ---------------------------------------------------
TRAINER_FILE = "nnUNetTrainer_batchsize.py"
_trainer_candidates = [
    os.path.join(WORKSPACE, "trainer files", TRAINER_FILE),
    os.path.join(RESULTS_ROOT, TRAINER_FILE),
    os.path.join(RESULTS_ROOT, "keep_batchC", TRAINER_FILE),
    os.path.join("/content/drive/MyDrive", TRAINER_FILE),
]
src_path = next((p for p in _trainer_candidates if os.path.exists(p)), None)
if src_path is None:
    print(f"{TRAINER_FILE} not found in any of:")
    for p in _trainer_candidates: print("   -", p)
    print("\nUpload it with (Mac):")
    print('  rclone copy "<local>/nnUNetTrainer_batchsize.py" "gdrive:nnU-Net Project/workspace/trainer files/" -P')
    raise FileNotFoundError(TRAINER_FILE)
shutil.copy(src_path, os.path.join(TRAINER_DIR, TRAINER_FILE))
print(f"injected {TRAINER_FILE}  (from {src_path})")
with open(src_path) as fh:
    _src = fh.read()
if "@classmethod" not in _src.split("def build_network_architecture")[0][-200:]:
    print("WARNING: build_network_architecture does not appear to be a @classmethod in this file -- re-upload the current file.")

# --- resolve each cell's trainer class and results folder --------------------------
dataset_results = os.path.join(RESULTS_ROOT, DATASET)
if not os.path.isdir(dataset_results):
    print(f"{DATASET} not found under RESULTS_ROOT. Contents:")
    for d in sorted(os.listdir(RESULTS_ROOT)): print("   -", d)
    raise FileNotFoundError(dataset_results)

all_folders = sorted(d for d in os.listdir(dataset_results) if os.path.isdir(os.path.join(dataset_results, d)))
print(f"\n{len(all_folders)} model folder(s) in {DATASET}:")
for d in all_folders: print("   -", d)

MODEL_DIRS, TRAINER_CLS = {}, {}
print("\nResolving cells:")
for label, tr in CELLS.items():
    cls = recursive_find_python_class(TRAINER_DIR, tr, TRAINER_PKG)
    hits = [d for d in all_folders if d.split("__")[0] == tr]
    folder = os.path.join(dataset_results, hits[0]) if len(hits) == 1 else None
    print(f"   {label:18s} class {'OK' if cls else 'NOT FOUND':9s} folder {'OK  ' if folder else 'NONE'} {hits}")
    if cls and folder:
        MODEL_DIRS[label] = folder; TRAINER_CLS[label] = cls
    elif len(hits) > 1:
        raise RuntimeError(f"{label}: ambiguous folders {hits}")

if not MODEL_DIRS:
    raise RuntimeError("no cell resolved -- check the trainer file was injected and the folders were copied from the pod")

# --- verify: build each network as the predictor will, then STRICT-LOAD a checkpoint ------
NORM_CLASSES = ("InstanceNorm3d", "BatchNorm3d", "GroupNorm")
print("\nArchitecture verification:")
bad = []
for label in MODEL_DIRS:
    mdir  = MODEL_DIRS[label]
    plans = PlansManager(load_json(join(mdir, "plans.json")))
    dj    = load_json(join(mdir, "dataset.json"))
    cm    = plans.get_configuration("3d_fullres")
    n_in  = len(dj.get("channel_names", dj.get("modality", {})))
    try:
        net = TRAINER_CLS[label].build_network_architecture(plans, dj, cm, n_in, enable_deep_supervision=True)
    except TypeError as e:
        print(f"   {label:18s} class-level build FAILED: {e}")
        print("      -> the trainer file is not the @classmethod version. Re-upload nnUNetTrainer_batchsize.py.")
        bad.append(label); continue

    norms = {k: 0 for k in NORM_CLASSES}; acts = {}
    for m in net.modules():
        n = type(m).__name__
        if n in norms: norms[n] += 1
        if n in ("LeakyReLU","ReLU","PReLU","SiLU","ELU","GELU","Mish","TanhExp","ELiSH","HardELiSH","Logish","Smish"):
            acts[n] = acts.get(n, 0) + 1
    want_norm = {"BN": "BatchNorm3d", "GN": "GroupNorm"}[NORM_OF[label]]
    print(f"   {label:18s} norms={norms}  acts={acts}  expected norm={want_norm}")

    ck_path = None
    for f in range(5):
        cand = os.path.join(mdir, f"fold_{f}", "checkpoint_final.pth")
        if os.path.exists(cand): ck_path = cand; break
    if ck_path is None:
        print(f"   {label:18s} no checkpoint_final.pth in any fold -- cannot verify"); bad.append(label); del net; continue
    ck = torch.load(ck_path, map_location="cpu")
    sd = ck.get("network_weights", ck)
    sd = {k[7:] if k.startswith("module.") else k: v for k, v in sd.items()}
    try:
        missing, unexpected = net.load_state_dict(sd, strict=False)
        if missing or unexpected:
            print(f"   {label:18s} STRICT LOAD FAILED  missing={len(missing)} unexpected={len(unexpected)}")
            for k in list(missing)[:5]:    print(f"        missing:    {k}")
            for k in list(unexpected)[:5]: print(f"        unexpected: {k}")
            bad.append(label)
        else:
            print(f"   {label:18s} checkpoint loads strictly OK  ({os.path.relpath(ck_path, mdir)}; keys={len(sd)}; running stats: {'yes' if any('running_mean' in k for k in sd) else 'no'})")
    except Exception as e:
        print(f"   {label:18s} LOAD ERROR {type(e).__name__}: {e}"); bad.append(label)
    del net, ck, sd
torch.cuda.empty_cache()
if bad:
    raise RuntimeError(f"checkpoint does not load into the built network for {bad} -- do not run inference")
print(f"\n{len(MODEL_DIRS)}/{len(CELLS)} cells resolved and verified by strict checkpoint load.")


## 2b — Fold audit (run before committing GPU hours)

Per activation: which folds exist, and whether each has `checkpoint_final.pth` (finished) or only
`checkpoint_latest.pth` (unfinished). A cell ensembled from unfinished folds is **not comparable**
to a complete 5-fold cell — the audit surfaces that before you spend hours on inference and submit
non-comparable numbers to Synapse.

In [ ]:
FOLD_INFO = {}
print(f"{'cell':18s} {'folds':>20s}  {'final':>5s} {'latest':>6s} {'missing':>7s}  status")
for act, mdir in MODEL_DIRS.items():
    final_f, latest_f, missing_f = [], [], []
    for fold in range(5):
        fd = os.path.join(mdir, f"fold_{fold}")
        if os.path.exists(os.path.join(fd, "checkpoint_final.pth")):
            final_f.append(fold)
        elif os.path.exists(os.path.join(fd, "checkpoint_latest.pth")):
            latest_f.append(fold)
        else:
            missing_f.append(fold)
    usable = sorted(final_f + latest_f)
    FOLD_INFO[act] = {"final": final_f, "latest": latest_f, "missing": missing_f, "usable": usable}
    if len(final_f) == 5:
        status = "COMPLETE"
    elif not usable:
        status = "NO CHECKPOINTS"
    elif latest_f:
        status = f"INCOMPLETE (folds {latest_f} unfinished)"
    else:
        status = f"PARTIAL ({len(usable)}/5 folds)"
    print(f"{act:18s} {str(usable):>20s}  {len(final_f):5d} {len(latest_f):6d} {len(missing_f):7d}  {status}")

ready    = [a for a, i in FOLD_INFO.items() if len(i["final"]) == 5]
degraded = [a for a, i in FOLD_INFO.items() if i["usable"] and len(i["final"]) < 5]
empty    = [a for a, i in FOLD_INFO.items() if not i["usable"]]

print(f"\n   fully complete (5/5 final): {ready}")
if degraded:
    print(f"   DEGRADED — will run but are not comparable: {degraded}")
if empty:
    print(f"   no checkpoints, will be skipped: {empty}")

# Set to True to run only the fully-complete cells (recommended for the reported table).
STRICT_5FOLD_ONLY = False
RUN_LIST = ready if STRICT_5FOLD_ONLY else ready + degraded
print(f"\n   RUN_LIST ({len(RUN_LIST)}): {RUN_LIST}")

## 3 — Stage the validation data (idempotent)

Copies the 219 raw validation cases into nnU-Net's `imagesTs` naming. Skips files already present,
so re-running is free. Channel map: `0000` T1 / `0001` T1c / `0002` T2 / `0003` FLAIR, accepting both
the legacy (`t1`, `t1ce`, `t2`, `flair`) and BraTS-2023 (`t1n`, `t1c`, `t2w`, `t2f`) suffixes.

In [ ]:
print("Staging validation data...")
if not os.path.isdir(RAW_VAL_DIR):
    raise FileNotFoundError(f"Raw validation folder not found: {RAW_VAL_DIR}")

patient_folders = [f.path for f in os.scandir(RAW_VAL_DIR) if f.is_dir()]
copied = 0
for pf in tqdm(patient_folders, desc="Copying & renaming"):
    pid = os.path.basename(pf)
    for fp in glob.glob(os.path.join(pf, "*.nii.gz")):
        fn = os.path.basename(fp).lower()
        cid = None
        if   fn.endswith("t1.nii.gz")  or fn.endswith("t1n.nii.gz"):   cid = "0000"
        elif fn.endswith("t1c.nii.gz") or fn.endswith("t1ce.nii.gz"):  cid = "0001"
        elif fn.endswith("t2.nii.gz")  or fn.endswith("t2w.nii.gz"):   cid = "0002"
        elif fn.endswith("flair.nii.gz") or fn.endswith("t2f.nii.gz"): cid = "0003"
        if cid:
            tgt = os.path.join(TARGET_IMAGES_TS, f"{pid}_{cid}.nii.gz")
            if not os.path.exists(tgt):
                shutil.copy(fp, tgt); copied += 1

n_files = len([f for f in os.listdir(TARGET_IMAGES_TS) if f.endswith(".nii.gz")])
NUM_CASES = n_files // 4
print(f"Staged {copied} new file(s). {n_files} files = {NUM_CASES} cases.")
if n_files % 4 != 0:
    print(f"   WARNING: {n_files} is not divisible by 4 — a modality may be missing for some case.")

## 4 — Inference: 5-fold soft-voting ensemble per cell

Predictor settings identical to every other cell (`tile_step_size=0.5`, Gaussian weighting,
mirroring on, `overwrite=False`) — a different setting would break the paired comparison with the
batch-2 counterparts. Resumable: a cell is skipped if its output folder already holds `NUM_CASES`
masks. Two cells × 5 folds × 219 cases — expect a few hours on a T4/L4, less on an A100.


In [ ]:
import os, time, traceback

def n_masks(folder):
    return len([f for f in os.listdir(folder) if f.endswith('.nii.gz')]) if os.path.isdir(folder) else 0

TODO = [a for a in LABELS if a in MODEL_DIRS and a in RUN_LIST]
print('cells to process:', TODO)
print('expected cases per activation:', NUM_CASES)

summary = []
for idx, act in enumerate(TODO, 1):
    mdir  = MODEL_DIRS[act]
    folds = FOLD_INFO[act]['usable']
    ckpt  = 'checkpoint_final.pth' if not FOLD_INFO[act]['latest'] else 'checkpoint_latest.pth'
    out   = os.path.join(PRED_ROOT, 'Validation_bs8_' + act)
    os.makedirs(out, exist_ok=True)

    print()
    print('=' * 66)
    print('  [%d/%d]  %s   folds=%s   ckpt=%s' % (idx, len(TODO), act, tuple(folds), ckpt))
    print('=' * 66)

    done = n_masks(out)
    if done >= NUM_CASES:
        print('  already complete (%d/%d) - skipping.' % (done, NUM_CASES))
        summary.append({'cell': act, 'folds': folds, 'ckpt': ckpt,
                        'masks': done, 'status': 'skipped (complete)'})
        continue
    if done:
        print('  resuming: %d/%d masks already present' % (done, NUM_CASES))

    t0 = time.time()
    try:
        predictor = nnUNetPredictor(
            tile_step_size=0.5, use_gaussian=True, use_mirroring=True,
            perform_everything_on_gpu=True,
            device=torch.device('cuda', 0),
            verbose=False, verbose_preprocessing=False, allow_tqdm=True,
        )
        predictor.initialize_from_trained_model_folder(
            mdir, use_folds=tuple(folds), checkpoint_name=ckpt)
        predictor.predict_from_files(
            TARGET_IMAGES_TS, out,
            save_probabilities=False,
            overwrite=False,
            num_processes_segmentation_export=1,
            folder_with_segs_from_prev_stage=None,
            num_parts=1, part_id=0,
        )
        del predictor; torch.cuda.empty_cache()
        got  = n_masks(out)
        mins = (time.time() - t0) / 60
        print('  done: %d/%d masks in %.0f min' % (got, NUM_CASES, mins))
        summary.append({'cell': act, 'folds': folds, 'ckpt': ckpt, 'masks': got,
                        'status': 'ok' if got == NUM_CASES else 'INCOMPLETE (%d/%d)' % (got, NUM_CASES),
                        'minutes': round(mins, 1)})
    except Exception as e:
        torch.cuda.empty_cache()
        print('  FAILED: %s: %s' % (type(e).__name__, e))
        traceback.print_exc(limit=5)
        summary.append({'cell': act, 'folds': folds, 'ckpt': ckpt,
                        'masks': n_masks(out), 'status': 'FAILED (%s)' % type(e).__name__})
        continue

print()
print('=' * 66)
print('INFERENCE SUMMARY')
print('=' * 66)
for s in summary:
    print('  %-18s %4d/%d  %-24s %s' % (s['cell'], s['masks'], NUM_CASES, s['ckpt'], s['status']))

## 5 — Package for Synapse (one zip per cell)

Casts labels to `uint8` and zips with bare case filenames, exactly as before. Resumable.


In [ ]:
import zipfile
import numpy as np
import nibabel as nib

OVERWRITE_ZIPS = False
zip_report = []

for act in RUN_LIST:
    out = os.path.join(PRED_ROOT, f"Validation_bs8_{act}")
    zp  = os.path.join(ZIP_ROOT, f"BraTS2023_Submission_bs8_{act}.zip")

    if not os.path.isdir(out):
        print(f"  {act:18s} no prediction folder — skipping."); continue
    files = [f for f in os.listdir(out) if f.endswith(".nii.gz")]
    if not files:
        print(f"  {act:18s} no masks — skipping."); continue
    if os.path.exists(zp) and not OVERWRITE_ZIPS:
        mb = os.path.getsize(zp) / 1e6
        print(f"  {act:18s} zip exists ({mb:.0f} MB) — skipping. Set OVERWRITE_ZIPS=True to rebuild.")
        zip_report.append({"activation": act, "n": len(files), "zip": zp, "status": "existing"})
        continue

    if len(files) != NUM_CASES:
        print(f"  {act:18s} WARNING: {len(files)} masks, expected {NUM_CASES} — zipping anyway.")

    with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as zf:
        for fn in tqdm(files, desc=f"{act:18s}", leave=False):
            img = nib.load(os.path.join(out, fn))
            data = img.get_fdata().astype(np.uint8)
            clean = nib.Nifti1Image(data, img.affine, img.header)
            clean.set_data_dtype(np.uint8)
            name = fn.replace("ensemble_pred_", "")
            tmp = os.path.join("/content", name)
            nib.save(clean, tmp)
            zf.write(tmp, arcname=name)
            os.remove(tmp)
    mb = os.path.getsize(zp) / 1e6
    print(f"  {act:18s} {len(files)} masks -> {os.path.basename(zp)} ({mb:.0f} MB)")
    zip_report.append({"activation": act, "n": len(files), "zip": zp, "status": "built"})

print(f"\n{len(zip_report)} zip(s) in {ZIP_ROOT}")

## 6 — Manifest

Writes `submission_manifest_bs8.csv` alongside the zips. Submit one zip at a time and record the
returned lesion-wise scores against this manifest.


In [ ]:
import os, glob
import pandas as pd

def n_masks(folder):
    return len([f for f in os.listdir(folder) if f.endswith(".nii.gz")]) if os.path.isdir(folder) else 0

# in-memory results if the inference cell ran in this session; otherwise fall back to disk
mem = {s["cell"]: s for s in summary} if "summary" in dir() and summary else {}
zips = {os.path.basename(p).replace("BraTS2023_Submission_bs8_", "").replace(".zip", ""): p
        for p in glob.glob(os.path.join(ZIP_ROOT, "BraTS2023_Submission_bs8_*.zip"))}

acts = sorted(set(list(MODEL_DIRS) if "MODEL_DIRS" in dir() else [])
              | set(mem) | set(zips))
if not acts:
    acts = sorted(os.path.basename(p).replace("Validation_bs8_", "")
                  for p in glob.glob(os.path.join(PRED_ROOT, "Validation_bs8_*")))

expected = NUM_CASES if "NUM_CASES" in dir() else None
rows = []
for a in acts:
    pred = os.path.join(PRED_ROOT, f"Validation_bs8_{a}")
    got  = n_masks(pred)
    fi   = FOLD_INFO.get(a, {}) if "FOLD_INFO" in dir() else {}
    used = fi.get("usable", [])
    ckpt = mem.get(a, {}).get("ckpt") or (
        "checkpoint_final.pth" if not fi.get("latest") else "checkpoint_latest.pth")
    zp   = zips.get(a, "")
    if got == 0:                      status = "not run"
    elif expected and got < expected: status = f"INCOMPLETE ({got}/{expected})"
    else:                             status = "ok"
    rows.append({
        "cell": a,
        "norm": NORM_OF.get(a, "?"),
        "batch_size_train": 8,
        "folds_used": ",".join(map(str, used)),
        "n_folds": len(used),
        "checkpoint": ckpt,
        "complete_5fold_final": len(fi.get("final", [])) == 5 if fi else None,
        "n_masks": got,
        "expected": expected,
        "status": status,
        "zip": os.path.basename(zp),
        "zip_MB": round(os.path.getsize(zp) / 1e6, 1) if zp else None,
    })

if not rows:
    print("Nothing to report — no predictions, no zips, no MODEL_DIRS.")
    print("Run the inference cell first.")
else:
    man = pd.DataFrame(rows).sort_values("activation")
    os.makedirs(ZIP_ROOT, exist_ok=True)
    man_path = os.path.join(ZIP_ROOT, "submission_manifest_bs8.csv")
    man.to_csv(man_path, index=False)
    print(man.to_string(index=False))
    print(f"\nmanifest -> {man_path}")

    ready = man[(man.status == "ok") & (man.zip != "")]
    print(f"\nready to submit: {len(ready)}/{len(man)}  {list(ready.cell)}")
    pend = man[man.status != "ok"]
    if len(pend):
        print("not ready:")
        for _, r in pend.iterrows():
            print(f"   {r.activation}: {r.status}")
    bad = man[man.complete_5fold_final == False]
    if len(bad):
        print("\nNOT comparable to a complete 5-fold cell — flag in any table:")
        for _, r in bad.iterrows():
            print(f"   {r.activation}: {r.n_folds} folds, {r.checkpoint}")

## 7 — After Synapse scoring

Download the per-case CSV for each cell and save it alongside the other results as

```
1st review/Synapse/BN_LeakyReLU_bs8.csv
1st review/Synapse/GN_LeakyReLU_bs8.csv
```

Those two files, paired against the existing batch-2 `BN_LeakyReLU` and `GN_LeakyReLU` per-case
CSVs on the same 219 patients, are the batch-size analysis: Wilcoxon signed-rank on each region and
metric, Holm-corrected. The reading rule was fixed before training:

| BN@8 vs BN@2 | GN@8 vs GN@2 | conclusion |
|---|---|---|
| improves | unchanged | BN's weakness *was* small-batch statistics |
| unchanged | unchanged | not a batch-size effect — the reported BN result stands |
| both improve | | the regime (4× fewer optimiser steps at batch 8), not normalisation |
| degrades | | undertrained at batch 8 — report as such |

Every outcome is an interpretable result.
